0. Imports and Setup

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from collections import Counter

# Sklearn 
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, cross_val_score, learning_curve
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, auc
)

# XGBoost 
from xgboost import XGBClassifier

# Reproducibility
np.random.seed(42)

print('Libraries loaded successfully.')

Libraries loaded successfully.


1. Load Preprocessed Data

In [ ]:
# Load data
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']        # SMOTE-balanced training features (PCA)
y_train = data['y_train']        # SMOTE-balanced training labels
X_val   = data['X_val']          # Validation features
y_val   = data['y_val']          # Validation labels
X_test  = data['X_test']         # Test features 
y_test  = data['y_test']         # Test labels
le      = data['label_encoder']  # LabelEncoder

n_classes = len(le.classes_)
class_names = list(le.classes_)

print('Data loaded successfully.')
print(f'X_train (SMOTE): {X_train.shape}   y_train: {Counter(y_train)}')
print(f'X_val:           {X_val.shape}   y_val:   {Counter(y_val)}')
print(f'X_test:          {X_test.shape}   y_test:  {Counter(y_test)}')
print(f'Classes ({n_classes}): {class_names}')

Data loaded successfully.
  X_train (SMOTE): (1695, 417)   y_train: Counter({np.int32(2): 339, np.int32(3): 339, np.int32(1): 339, np.int32(0): 339, np.int32(4): 339})
  X_val:           (118, 417)   y_val:   Counter({np.int32(2): 60, np.int32(3): 24, np.int32(0): 21, np.int32(1): 9, np.int32(4): 4})
  X_test:          (197, 417)   y_test:  Counter({np.int32(2): 100, np.int32(3): 40, np.int32(0): 34, np.int32(1): 16, np.int32(4): 7})
  Classes (5): ['Basal', 'Her2', 'LumA', 'LumB', 'Normal']


3. Model 1 — Bagged Elastic Net 

3.1 Hyperparameter Tuning — Bagged Elastic Net

In [ ]:
# agged Elastic Net: Logistic Regression with ElasticNet penalty 

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier

# Define base estimator 
base_en = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    max_iter=1000,
    random_state=42
)

# Hyperparameter grid 
param_grid_ben = {
    'estimator__C':      [0.01, 0.1, 1.0, 10.0],   # Inverse regularization strength
    'estimator__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],  # Mix of L1 vs L2
    'n_estimators':      [10, 20, 30],               # Number of bagged learners
    'max_samples':       [0.7, 0.8, 1.0],            # Bootstrap sample fraction
}

bagging_en = BaggingClassifier(
    estimator=base_en,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_ben = GridSearchCV(
    bagging_en,
    param_grid_ben,
    cv=cv_strat,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Running GridSearchCV for Bagged Elastic Net')
print(f'Total combinations: {4 * 5 * 3 * 3} × 5 folds = {4 * 5 * 3 * 3 * 5} fits')
grid_ben.fit(X_train, y_train)

print(f'\nBest parameters: {grid_ben.best_params_}')
print(f'Best CV F1-macro: {grid_ben.best_score_:.4f}')
best_ben = grid_ben.best_estimator_

Running GridSearchCV for Bagged Elastic Net...
Total combinations: 180 × 5 folds = 900 fits
Fitting 5 folds for each of 180 candidates, totalling 900 fits

Best parameters: {'estimator__C': 10.0, 'estimator__l1_ratio': 0.9, 'max_samples': 1.0, 'n_estimators': 30}
Best CV F1-macro: 0.9720


In [ ]:
# Validation set check 
y_val_pred_ben = best_ben.predict(X_val)
val_f1_ben = f1_score(y_val, y_val_pred_ben, average='macro', zero_division=0)
print(f'Bagged Elastic Net - Validation F1-macro: {val_f1_ben:.4f}')

Bagged Elastic Net — Validation F1-macro: 0.8615


4. Model 2 — Support Vector Machine (SVM)

4.1 Hyperparameter Tuning — SVM

In [ ]:
param_grid_svm = {
    'C':      [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma':  ['scale', 'auto'],      
}

svm_base = SVC(probability=True, random_state=42, decision_function_shape='ovr')

grid_svm = GridSearchCV(
    svm_base,
    param_grid_svm,
    cv=cv_strat,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Running GridSearchCV for SVM')
print(f'Total combinations: {4 * 3 * 2} × 5 folds = {4 * 3 * 2 * 5} fits')
grid_svm.fit(X_train, y_train)

print(f'\nBest parameters: {grid_svm.best_params_}')
print(f'Best CV F1-macro: {grid_svm.best_score_:.4f}')
best_svm = grid_svm.best_estimator_

Running GridSearchCV for SVM...
Total combinations: 24 × 5 folds = 120 fits
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best parameters: {'C': 0.1, 'gamma': 'auto', 'kernel': 'poly'}
Best CV F1-macro: 0.9787


In [8]:
y_val_pred_svm = best_svm.predict(X_val)
val_f1_svm = f1_score(y_val, y_val_pred_svm, average='macro', zero_division=0)
print(f'SVM — Validation F1-macro: {val_f1_svm:.4f}')

SVM — Validation F1-macro: 0.8689


5. Model 3 — Random Forest

5.1 Hyperparameter Tuning — Random Forest

In [10]:
param_grid_rf = {
    'n_estimators':     [50, 100, 200, 500],
    'max_depth':        [None, 10, 20, 30],
    'min_samples_split':[2, 5, 10],
    'max_features':     ['sqrt', 'log2'],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')

grid_rf = GridSearchCV(
    rf_base,
    param_grid_rf,
    cv=cv_strat,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Running GridSearchCV for Random Forest')
print(f'Total combinations: {4 * 4 * 3 * 2} × 5 folds = {4 * 4 * 3 * 2 * 5} fits')
grid_rf.fit(X_train, y_train)

print(f'\nBest parameters: {grid_rf.best_params_}')
print(f'Best CV F1-macro: {grid_rf.best_score_:.4f}')
best_rf = grid_rf.best_estimator_

Running GridSearchCV for Random Forest...
Total combinations: 96 × 5 folds = 480 fits
Fitting 5 folds for each of 96 candidates, totalling 480 fits

Best parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 500}
Best CV F1-macro: 0.9752


In [ ]:
y_val_pred_rf = best_rf.predict(X_val)
val_f1_rf = f1_score(y_val, y_val_pred_rf, average='macro', zero_division=0)
print(f'Random Forest - Validation F1-macro: {val_f1_rf:.4f}')

Random Forest — Validation F1-macro: 0.6370


6. Model 4 — XGBoost

6.1 Hyperparameter Tuning — XGBoost

In [ ]:
param_grid_xgb = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample':     [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=n_classes,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

from sklearn.model_selection import RandomizedSearchCV
grid_xgb = RandomizedSearchCV(
    xgb_base,
    param_grid_xgb,
    n_iter=30,       
    cv=cv_strat,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=True
)

print('Running RandomizedSearchCV for XGBoost')
grid_xgb.fit(X_train, y_train)

print(f'\nBest parameters: {grid_xgb.best_params_}')
print(f'Best CV F1-macro: {grid_xgb.best_score_:.4f}')
best_xgb = grid_xgb.best_estimator_

Running RandomizedSearchCV for XGBoost (30 iterations × 5 folds = 150 fits)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
Best CV F1-macro: 0.9672


In [ ]:
y_val_pred_xgb = best_xgb.predict(X_val)
val_f1_xgb = f1_score(y_val, y_val_pred_xgb, average='macro', zero_division=0)
print(f'XGBoost - Validation F1-macro: {val_f1_xgb:.4f}')

XGBoost — Validation F1-macro: 0.8044


7. Save All Trained Models

In [ ]:
models_output = {
    'Bagged_Elastic_Net': best_ben,
    'SVM':                best_svm,
    'Random_Forest':      best_rf,
    'XGBoost':            best_xgb,
    'results_df':         df_results,
    'tuning_df':          df_all_tuning,
    'cv_scores':          cv_scores,
    'class_names':        class_names,
    'label_encoder':      le,
}

with open('trained_models.pkl', 'wb') as f:
    pickle.dump(models_output, f)
    
print('All trained models saved to trained_models.pkl')

All trained models saved to trained_models.pkl
